In [1]:
import nltk
import pandas as pd
from gensim.models import Word2Vec
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pravi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
sentences, domains = [], []
current_domain = None
for line in open('corpus.txt'):
    line = line.strip()
    if line in ("MEDICAL", "LEGAL"):
        current_domain = line.lower()
    elif line:
        sentences.append(line)
        domains.append(current_domain)
df = pd.DataFrame({'sentences': sentences, 'domain': domains})
print(f'total sentences: {len(df)}')
print(f'Medical sentences: {len(df[df.domain=="medical"])}')
print(f'Legal sentences: {len(df[df.domain=="legal"])}')

total sentences: 80
Medical sentences: 40
Legal sentences: 40


In [3]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
corpus = [
    [w for w in sentence.lower().split() if w not in stop_words and len(w)>2]
    for sentence in df['sentences']
]
corpus=[token for token in corpus if token]
print('before cleaning:',df['sentences'])
print('after cleaning:',corpus)

before cleaning: 0       doctor treats patient in hospital with medicine
1            nurse administers medicine to sick patient
2          surgeon performs complex surgery in hospital
3     doctor prescribes antibiotics for bacterial in...
4     patient needs insulin injection for diabetes m...
                            ...                        
75    attorney challenges witness credibility in cri...
76    judge grants plaintiff request for damages in ...
77    criminal defense lawyer negotiates plea deal w...
78    defendant cross examined by plaintiff attorney...
79    jury finds defendant guilty of criminal charge...
Name: sentences, Length: 80, dtype: object
after cleaning: [['doctor', 'treats', 'patient', 'hospital', 'medicine'], ['nurse', 'administers', 'medicine', 'sick', 'patient'], ['surgeon', 'performs', 'complex', 'surgery', 'hospital'], ['doctor', 'prescribes', 'antibiotics', 'bacterial', 'infection'], ['patient', 'needs', 'insulin', 'injection', 'diabetes', 'managemen

In [ ]:
model = Word2Vec(corpus, vector_size=100, window=4, min_count=2,sg=1,seed=42)
print('vocabulary size:',len(model.wv))
print('vector size:',model.vector_size)

vocabulary size: 64
vector size: 100


In [8]:
# --- 1. Cosine Similarity between two words ---
word1, word2 = 'doctor', 'patient'
similarity = model.wv.similarity(word1, word2)
print(f'Cosine Similarity between "{word1}" and "{word2}": {similarity:.4f}')

word3, word4 = 'court', 'judge'
similarity2 = model.wv.similarity(word3, word4)
print(f'Cosine Similarity between "{word3}" and "{word4}": {similarity2:.4f}')

# --- 2. Top 3 similar words for a given word ---
for word in ['doctor', 'court', 'patient']:
    top3 = model.wv.most_similar(word, topn=3)
    print(f'\nTop 3 words similar to "{word}":')
    for w, score in top3:
        print(f'  {w}: {score:.4f}')

# --- 3. Vector Arithmetic ---
# Example: doctor - patient + defendant ≈ ?
result = model.wv.most_similar(positive=['doctor', 'defendant'], negative=['patient'], topn=3)
print('\nVector Arithmetic: doctor - patient + defendant ≈')
for w, score in result:
    print(f'  {w}: {score:.4f}')

# Example: court - judge + hospital ≈ ?
result2 = model.wv.most_similar(positive=['court', 'hospital'], negative=['judge'], topn=3)
print('\nVector Arithmetic: court - judge + hospital ≈')
for w, score in result2:
    print(f'  {w}: {score:.4f}')

Cosine Similarity between "doctor" and "patient": -0.0069
Cosine Similarity between "court" and "judge": -0.1860

Top 3 words similar to "doctor":
  verdict: 0.3155
  lawyer: 0.2342
  monitors: 0.2077

Top 3 words similar to "court":
  medicine: 0.2058
  prescribes: 0.1943
  murder: 0.1828

Top 3 words similar to "patient":
  medicine: 0.2153
  management: 0.2026
  hospital: 0.1937

Vector Arithmetic: doctor - patient + defendant ≈
  verdict: 0.3187
  diagnoses: 0.2226
  attorney: 0.1604

Vector Arithmetic: court - judge + hospital ≈
  medicine: 0.2805
  cross: 0.1829
  verdict: 0.1798
